# 06 — Viterbi / sequence model

- Inputs: engineered observations and algorithmic anchors.
- Sequence model: chooses a coherent path through H/A/D/X states.
- Viterbi smoothing is a decoder, not new evidence.
- It cannot repair a missing or ambiguous touch by itself.
- Hard evidence protects outside/dead frames; uncertain transitions remain conservative.
- Compare the decoded path with the evidence window and public GT; do not treat the path as ground truth.

## Worked dynamic-programming example

Viterbi decoding selects the most coherent state path, frame by frame. For state `s` at frame `t`, the recurrence is:

`V[t, s] = emission[t, s] + max(previous V[t-1, s_prev] + transition[s_prev, s])`

- The emission score says how well the observations support `H`, `A`, or `D` at this frame.
- The transition score rewards continuity and penalises implausible rapid switching.
- The backpointer stores which previous state produced the best score.
- Backtracking from the best final state gives the complete decoded path.

This toy window is deliberately small so the calculation is inspectable; the production model applies the same dynamic-programming structure to the full engineered feature table.

In [1]:
import pandas as pd
import numpy as np

states = ["H", "D", "A"]
frames = [112, 113, 114, 115, 116]
emission = np.array([
    [3.0, 0.2, 0.0],  # frame 112: H evidence
    [2.4, 0.4, 0.1],  # frame 113: H still plausible
    [0.5, 1.8, 0.6],  # frame 114: contested evidence
    [0.2, 1.0, 2.5],  # frame 115: A becomes plausible
    [0.0, 0.3, 3.0],  # frame 116: A evidence
])
transition = np.full((3, 3), -1.5)
np.fill_diagonal(transition, 0.0)

dp = np.full_like(emission, -np.inf)
back = np.full(emission.shape, -1, dtype=int)
dp[0] = emission[0]
for t in range(1, len(frames)):
    for s in range(len(states)):
        candidates = dp[t - 1] + transition[:, s]
        back[t, s] = int(np.argmax(candidates))
        dp[t, s] = emission[t, s] + candidates[back[t, s]]

path_idx = [int(np.argmax(dp[-1]))]
for t in range(len(frames) - 1, 0, -1):
    path_idx.append(int(back[t, path_idx[-1]]))
path_idx = path_idx[::-1]
worked = pd.DataFrame({
    "frame": frames,
    "emission_H": emission[:, 0],
    "emission_D": emission[:, 1],
    "emission_A": emission[:, 2],
    "best_H": dp[:, 0],
    "best_D": dp[:, 1],
    "best_A": dp[:, 2],
    "backpointer": ["—" if t == 0 else states[back[t, path_idx[t]]] for t in range(len(frames))],
    "selected_state": [states[i] for i in path_idx],
})
display(worked)
print("decoded path:", " → ".join(states[i] for i in path_idx))


,frame,emission_H,emission_D,emission_A,best_H,best_D,best_A,backpointer,selected_state
0,112,3.0,0.2,0.0,3.0,0.2,0.0,—,H
1,113,2.4,0.4,0.1,5.4,1.9,1.6,H,H
2,114,0.5,1.8,0.6,5.9,5.7,4.5,H,A
3,115,0.2,1.0,2.5,6.1,6.7,7.0,A,A
4,116,0.0,0.3,3.0,6.1,7.0,10.0,A,A


decoded path: H → H → A → A → A


In [2]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'possession').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from possession.baseline_pipeline import ClosestTeamRuleModel
from possession.calculations.features import build_engineered_features
from possession.sequence_models import AnchorInitializedHMM, AnchorInitializedHSMM
from possession.temporal_rules import load_rules

features = build_engineered_features(model=ClosestTeamRuleModel(), rules=load_rules())
hmm_labels = AnchorInitializedHMM(load_rules()).fit(features).predict(features)
hsmm_labels = AnchorInitializedHSMM(load_rules()).fit(features).predict(features)
comparison = pd.DataFrame({'frame':features.frame, 'hmm':hmm_labels, 'hsmm':hsmm_labels, 'anchor':features.anchor_team, 'scenario':features.scenario_context})
display(comparison.head())


,frame,hmm,hsmm,anchor,scenario
0,0,X,X,NaN,OCCLUDED
1,1,X,X,NaN,OCCLUDED
2,2,X,X,NaN,OCCLUDED
3,3,X,X,NaN,OCCLUDED
4,4,X,X,NaN,OCCLUDED


## Worked example from our RuleConstrainedViterbi model

The following trace uses the actual `RuleConstrainedViterbi` implementation on the engineered match table. It exposes the model’s real quantities rather than replacing them with a toy example:

- emissions combine distance-margin evidence, rule seeds, Smart Tagging dead-ball evidence, and anchors;
- the transition matrix strongly favours staying in the current state;
- dynamic programming retains the best cumulative score for each state at every frame;
- backtracking converts those stored decisions into the selected H/A/D path.


In [3]:
from possession.sequence_models import RuleConstrainedViterbi

viterbi = RuleConstrainedViterbi(load_rules()).fit(features)
emission = viterbi._emission(features)
transition = viterbi.transition_logp
states = np.asarray(["H", "A", "D"])
score = np.empty_like(emission)
back = np.zeros(emission.shape, dtype=np.int8)
score[0] = viterbi.start_logp + emission[0]
for frame in range(1, len(features)):
    candidates = score[frame - 1][:, None] + transition
    back[frame] = np.argmax(candidates, axis=0)
    score[frame] = candidates[back[frame], np.arange(3)] + emission[frame]
path = np.empty(len(features), dtype=np.int8)
state = int(np.argmax(score[-1]))
for frame in range(len(features) - 1, -1, -1):
    path[frame] = state
    state = int(back[frame, state]) if frame else state
decoded = states[path]
switches = np.flatnonzero(decoded[1:] != decoded[:-1]) + 1
centre = int(switches[0]) if len(switches) else len(features) // 2
left, right = max(0, centre - 3), min(len(features), centre + 4)
trace = pd.DataFrame({
    "frame": features["frame"].iloc[left:right].to_numpy(),
    "emission_H": emission[left:right, 0],
    "emission_A": emission[left:right, 1],
    "emission_D": emission[left:right, 2],
    "best_path_score_H": score[left:right, 0],
    "best_path_score_A": score[left:right, 1],
    "best_path_score_D": score[left:right, 2],
    "selected_state": decoded[left:right],
})
display(trace)
print("actual model path around first decoded switch:", " → ".join(decoded[left:right]))


,frame,emission_H,emission_A,emission_D,best_path_score_H,best_path_score_A,best_path_score_D,selected_state
0,4196,-0.740568,0.740568,0.0,-25.232872,-17.765285,-22.131238,A
1,4197,-0.640720,0.640720,0.0,-24.397470,-17.129577,-22.136250,A
2,4198,-0.498188,0.498188,0.0,-23.619229,-16.636402,-22.141263,A
3,4199,0.439424,-0.439424,0.0,-22.188442,-17.080839,-22.146275,H
4,4200,0.593325,-0.593325,0.0,-21.600130,-17.679176,-22.151288,H
5,4201,0.756944,-0.756944,0.0,-20.848199,-18.441132,-22.156300,H
6,4202,0.930975,-0.930975,0.0,-19.922236,-19.377120,-22.161313,H


actual model path around first decoded switch: A → A → A → H → H → H → H
